# Ranking Prediction of the Regular Season For Each Conference

In [ ]:
from IPython.utils import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import spearmanr, kendalltau
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import ElasticNet


with io.capture_output() as captured:
    %run data_preprocessing.ipynb

## Data Integration

For the conference ranking prediction task, we integrate:
1. **Teams data** - core team performance metrics
3. **Player aggregated stats** - team roster quality indicators

The integration strategy:
- Start with `teams_df` as the base (one row per team-season)
- Add aggregated player statistics per team-season
- Create target variable: next year's win percentage

In [ ]:
# Sort teams by tmID and year to prepare for shifting
teams_ranking_df = teams_df.sort_values(['tmID', 'year']).copy()

# Verify teams don't change conferences
conf_changes = teams_ranking_df.groupby('tmID')['confID'].nunique()
teams_with_conf_changes = conf_changes[conf_changes > 1]
if len(teams_with_conf_changes) > 0:
    print(f"{len(teams_with_conf_changes)} teams changed conferences!")
    print(teams_with_conf_changes)

# Create target: next year's win percentage
teams_ranking_df['next_year_win_pct'] = teams_ranking_df.groupby('tmID')['win_pct'].shift(-1)

# Keep only rows where we have next year's win percentage
teams_ranking_df = teams_ranking_df.dropna(subset=['next_year_win_pct'])

print(f"Base dataset shape: {teams_ranking_df.shape}")
print(f"Seasons covered: {teams_ranking_df['year'].min()} to {teams_ranking_df['year'].max()}")

In [ ]:
# Aggregate player statistics per team-season
# This represents overall roster quality and depth

player_agg_stats = players_teams_df.groupby(['tmID', 'year']).agg({
    # Per-game averages across roster
    'ppg': 'mean',
    'apg': 'mean',
    'rpg': 'mean',
    'spg': 'mean',
    'bpg': 'mean',
    'mpg': 'mean',
    
    # Efficiency metrics
    'ts_pct': 'mean',
    'efg_pct': 'mean',
    'ast_to': 'mean',
    'sbdRpg': 'mean',
    
    # Roster depth indicators
    'playerID': 'count',  # number of players used
    'GP': 'mean'  # average games played (roster stability)
}).reset_index()

player_agg_stats = player_agg_stats.rename(columns={
    'playerID': 'roster_size',
    'GP': 'avg_player_gp',
    'ppg': 'roster_avg_ppg',
    'apg': 'roster_avg_apg',
    'rpg': 'roster_avg_rpg',
    'spg': 'roster_avg_spg',
    'bpg': 'roster_avg_bpg',
    'mpg': 'roster_avg_mpg',
    'ts_pct': 'roster_avg_ts',
    'efg_pct': 'roster_avg_efg',
    'ast_to': 'roster_avg_ast_to',
    'sbdRpg': 'roster_avg_sbdR'
})

# Merge player aggregates into main dataset
teams_ranking_df = teams_ranking_df.merge(player_agg_stats, on=['tmID', 'year'], how='left')

print(f"\nAfter player integration: {teams_ranking_df.shape}")
print(f"Missing player data: {teams_ranking_df['roster_size'].isna().sum()} rows")

The name and arena columns have no predictive value.

In [ ]:
teams_ranking_df = teams_ranking_df.drop(columns=['name', 'arena'], errors='ignore')

In [ ]:
# Final dataset summary
print("\nIntegrated Dataset Summary")
print(f"Total rows: {len(teams_ranking_df)}")
print(f"Total features: {len(teams_ranking_df.columns)}")

print(f"\nTarget variable distribution:")
display(teams_ranking_df['next_year_win_pct'].describe())

In [ ]:
print("\nSample of integrated data:")
display(teams_ranking_df.head(10))

Convert playoff column to binary indicator.

In [ ]:
teams_ranking_df['playoff'] = (teams_ranking_df['playoff'] == 'Y').astype(int)

In [ ]:
display(teams_ranking_df.head(10))

In [ ]:
display(teams_ranking_df.describe())
teams_ranking_df.shape

Remove roster_avg_ast_to due to infinity and NaN values.

In [ ]:
teams_ranking_df.drop(columns=['roster_avg_ast_to'], inplace=True)

## Prediction

In [ ]:
def evaluate_ranking_predictions(test_df, y_test, y_pred, display_metrics: bool = True):
    """
    Evaluate model by comparing predicted rankings vs actual rankings within conferences.

    Args:
        test_df: Test dataframe with tmID, confID, year
        y_test: Actual next year win percentages
        y_pred: Predicted next year win percentages
        display_metrics: Whether to display the per-conference metrics table
    """
    # Create evaluation dataframe
    eval_df = test_df[['tmID', 'confID', 'year']].copy()
    eval_df['actual_win_pct'] = y_test.values
    eval_df['predicted_win_pct'] = y_pred
    
    # Calculate actual and predicted rankings within each conference-year
    eval_df['actual_rank'] = eval_df.groupby(['confID', 'year'])['actual_win_pct'].rank(
        ascending=False, method='min'
    )
    eval_df['predicted_rank'] = eval_df.groupby(['confID', 'year'])['predicted_win_pct'].rank(
        ascending=False, method='min'
    )
    
    # Per-conference breakdown
    conf_results = []
    for conf in eval_df['confID'].unique():
        conf_years = eval_df[eval_df['confID'] == conf].groupby('year')
        mae_list = []
        spearman_list = []
        kendall_list = []
        teams_list = []
        
        for year, group in conf_years:
            mae = mean_absolute_error(group['actual_rank'], group['predicted_rank'])
            if len(group) > 1:
                spearman, _ = spearmanr(group['actual_rank'], group['predicted_rank'])
                kendall, _ = kendalltau(group['actual_rank'], group['predicted_rank'])
            else:
                spearman = np.nan
                kendall = np.nan
            mae_list.append(mae)
            spearman_list.append(spearman)
            kendall_list.append(kendall)
            teams_list.append(len(group))
        
        conf_results.append({
            'Conference': conf,
            'MAE': float(np.mean(mae_list)) if len(mae_list) else np.nan,
            'Spearman': float(np.nanmean(spearman_list)) if len(spearman_list) else np.nan,
            'Kendall': float(np.nanmean(kendall_list)) if len(kendall_list) else np.nan,
            'Teams': float(np.mean(teams_list)) if len(teams_list) else np.nan,
        })
    
    conf_metrics = pd.DataFrame(conf_results).round(3)
    if display_metrics:
        display(conf_metrics)
    
    return conf_metrics

In [ ]:
# Prepare features and target
feature_cols = [col for col in teams_ranking_df.columns
                if col not in ['next_year_win_pct', 'tmID', 'confID', 'year']]
X = teams_ranking_df[feature_cols]
y = teams_ranking_df['next_year_win_pct']

# Build year-based splits: train on all years <= split_year, test on next year
years = np.sort(teams_ranking_df['year'].unique())

# choose last 5 transitions as folds
n_splits = 5
possible_split_years = years[:-1]  # last year can't be a split_year (no next year to test)
split_points = possible_split_years[-n_splits:]

fold_metrics = {
    'Baseline (This Year win_pct)': [],
    'Linear Regression': [],
    'Random Forest': [],
    'Support Vector Machine': [],
    'ElasticNet': [],
}

for fold, split_year in enumerate(split_points, start=1):
    train_idx = teams_ranking_df.index[teams_ranking_df['year'] <= split_year]
    test_idx  = teams_ranking_df.index[teams_ranking_df['year'] == (split_year + 1)]

    X_train, X_test = X.loc[train_idx], X.loc[test_idx]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]

    # Baseline: predict next year's win% as this year's win% (persistence baseline)
    baseline_pred = teams_ranking_df.loc[test_idx, 'win_pct'].astype(float).to_numpy()
    fold_metrics['Baseline (This Year win_pct)'].append(
        evaluate_ranking_predictions(teams_ranking_df.loc[test_idx], y_test, baseline_pred, display_metrics=False)
    )

    # Feature selection
    selector = VarianceThreshold(threshold=0.01)
    selector.fit(X_train)
    selected_features = X_train.columns[selector.get_support()].tolist()

    X_train_temp = X_train[selected_features]
    correlations = X_train_temp.corrwith(y_train).abs().sort_values(ascending=False)
    top_features = correlations.head(40).index.tolist()

    corr_matrix = X_train_temp[top_features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
    final_features = [f for f in top_features if f not in to_drop]

    X_train_final = X_train[final_features]
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rfe = RFE(estimator=rf, n_features_to_select=8, step=1)
    rfe.fit(X_train_final, y_train)
    selected_rfe_features = X_train_final.columns[rfe.support_].tolist()

    X_train_fold = X_train[selected_rfe_features]
    X_test_fold = X_test[selected_rfe_features]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_fold)
    X_test_scaled = scaler.transform(X_test_fold)

    # Train and evaluate models
    lr_model = LinearRegression()
    lr_model.fit(X_train_scaled, y_train)
    y_lr_pred = lr_model.predict(X_test_scaled)
    fold_metrics['Linear Regression'].append(
        evaluate_ranking_predictions(teams_ranking_df.loc[test_idx], y_test, y_lr_pred, display_metrics=False)
    )

    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train_fold, y_train)
    y_rf_pred = rf_model.predict(X_test_fold)
    fold_metrics['Random Forest'].append(
        evaluate_ranking_predictions(teams_ranking_df.loc[test_idx], y_test, y_rf_pred, display_metrics=False)
    )

    svr_model = SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale')
    svr_model.fit(X_train_scaled, y_train)
    y_svr_pred = svr_model.predict(X_test_scaled)
    fold_metrics['Support Vector Machine'].append(
        evaluate_ranking_predictions(teams_ranking_df.loc[test_idx], y_test, y_svr_pred, display_metrics=False)
    )

    enet_model = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42, max_iter=10000)
    enet_model.fit(X_train_scaled, y_train)
    y_enet_pred = enet_model.predict(X_test_scaled)
    fold_metrics['ElasticNet'].append(
        evaluate_ranking_predictions(teams_ranking_df.loc[test_idx], y_test, y_enet_pred, display_metrics=False)
    )

    

In [ ]:
# Mean metrics across all splits (overall, averaged over conference tables)
summary_rows = []
for model_name, metrics_list in fold_metrics.items():
    if not metrics_list:
        continue
    all_metrics = pd.concat(metrics_list, ignore_index=True)
    means = all_metrics[['MAE', 'Spearman', 'Kendall']].mean(numeric_only=True)
    summary_rows.append({
        'Model': model_name,
        'MAE': means['MAE'],
        'Spearman': means['Spearman'],
        'Kendall': means['Kendall'],
    })

summary_df = pd.DataFrame(summary_rows).round(3)
display(summary_df)